In [1]:
import os
import openai

OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")

# 클라이언트 초기화
client = openai.OpenAI(
    base_url=OLLAMA_API_URL, # OpenAI API 대신 ollama 사용
    api_key="ollama-key" # 필수로 문자열을 요구하므로 주석 해제 유지
)

# Ollama 서버의 모델 목록 조회
models = client.models.list()

# 사용 가능한 모델의 ID(이름) 출력
print("=== 사용 가능한 모델 목록 ===")
for model in models.data:
    print("-", model.id)


=== 사용 가능한 모델 목록 ===
- gemma4:e4b
- gemma4:e2b
- qwen3.5:4b
- qwen3.5:9b
- nemotron-3-nano:4b
- functiongemma:latest
- translategemma:12b
- translategemma:4b
- nomic-embed-text-v2-moe:latest
- olmo-3:7b
- ministral-3:14b
- ministral-3:8b
- ministral-3:3b
- qwen3-vl:8b
- qwen3-vl:4b
- qwen3-vl:2b
- qwen3-embedding:0.6b
- embeddinggemma:300m
- gemma3n:e2b
- gemma3n:e4b
- linux6200/bge-reranker-v2-m3:latest
- gemma3:4b
- gemma3:1b
- gemma3:12b
- bona/bge-m3-korean:latest
- shieldgemma:latest
- llama-guard3:8b
- llama3.2-vision:11b
- nomic-embed-text:latest
- codegemma:latest


In [2]:
import openai
import json
import requests

def get_weather(city):
    return "33도의 더운 날씨"

# 함수 이름(문자열)과 실제 파이썬 함수를 매핑해주는 딕셔너리
FUNCTION_MAP = {
    "get_weather": get_weather,
}

# 2. 클라이언트 초기화
client = openai.OpenAI(
    base_url=OLLAMA_API_URL,
    api_key="ollama-key" 
)

# 3. 도구(Tools) 정의 (이전과 동일)
TOOLS = [
     {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "해당 도시의 날씨를 가져옵니다",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "text", "description": "도시 영문명"}}
            }
        }
    },
]

# 4. 메모리
messages = []

In [ ]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role":"assistant", "content": message.content or "",
            "tool_calls": [{
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments,
                },
            } for tool_call in message.tool_calls]
        })
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")
            
            try: 
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)
            print(f"🛠️ [TOOL]: {result}")

            messages.append({"role":"tool", "tool_call_id": tool_call.id, "name": function_name, "content": result})
        call_llm() # TOOL 실행결과를 담아서 다시 요청

    else: 
        llm_message = message.content
    
        print(f"🤖 [답변]: {llm_message}")
        messages.append({"role":"assistant", "content": llm_message})

In [4]:
def call_llm():
    response = client.chat.completions.create(
        model="gemma4:e2b",
        messages=messages,
        tools=TOOLS,
        tool_choice="auto"
    )

    print(response)
    
    process_ai_response(response.choices[0].message)


In [5]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        print(f"\n👤 [사용자 입력]: \"{message}\"")
        messages.append({"role":"user", "content": message})
        call_llm()


👤 [사용자 입력]: "hi"
ChatCompletion(id='chatcmpl-388', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I help you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='1.  **Analyze the user\'s input:** The user said "hi".\n2.  **Determine the user\'s intent:** This is a very general greeting. It doesn\'t ask a question, request information, or command an action that requires tool use.\n3.  **Review available tools:** The only available tool is `get_weather`, which requires a `city` parameter.\n4.  **Evaluate tool applicability:** The input "hi" does not provide any information that can be used to call the `get_weather` tool (no city specified).\n5.  **Formulate a response:** Since the input is just a greeting, the appropriate response is a friendly acknowledgment or continuation of the conversation.\n6.  **Final response generation:** Respond naturally to t